# 第二部分：ONNX 部署详解（使用者指南）

ONNX 是**跨框架模型交换格式**。  
对使用者来说，核心就三件事：  
**1. 把 PyTorch 模型导出为 ONNX**  
**2. 用 ONNX Runtime 加载并推理（Python / C++）**  
**3. 按需做一些性能优化（图优化、多线程等）**

---

## 一、整体流程

```text
PyTorch 训练好的模型
        │  torch.onnx.export()
        ▼
   model.onnx
        │
        │ onnxruntime (Python / C++)
        ▼
    推理结果
```

---

## 二、ONNX 简介（最简概念）

- ONNX = Open Neural Network Exchange，开放神经网络交换格式
- 本质：用 Protobuf 序列化的**计算图**（节点、权重、输入输出）
- 好处：一次导出，可以在 CPU、GPU、手机、嵌入式等多种后端运行

文件内部结构（了解即可）：
```text
model.onnx
├── graph (计算图)
│   ├── node (算子节点，如 Conv、Relu)
│   ├── initializer (权重)
│   ├── input / output
└── opset_import (算子集版本)
```

---

## 三、导出 ONNX（Python 端）

### 3.1 基础导出 API
```python
model.eval()   # 必须！否则 BN/Dropout 行为异常

dummy = torch.randn(1, 3, 224, 224)   # 示例输入（决定 shape 和 dtype）

torch.onnx.export(
    model,                       # 模型（eval 模式）
    dummy,                       # 示例输入
    "model.onnx",                # 输出文件
    opset_version=17,            # 算子集版本，常用 13 或 17
    input_names=["input"],       # 自定义输入名（推理时使用）
    output_names=["output"],     # 自定义输出名
    do_constant_folding=True,    # 常量折叠优化（建议开启）
    verbose=False
)
```

**核心参数说明**：
- `opset_version`：版本越高，支持的算子越多，但需推理后端兼容。一般 ≥ 13。
- `input_names / output_names`：自定义名称，后续用 ONNX Runtime 时以此名称喂入/取出数据。
- `do_constant_folding`：导出时预先计算常量表达式，减小图体积、加速推理。

### 3.2 动态 batch size（强烈推荐）
生产环境 batch 大小通常不固定，导出时需标记动态轴：

```python
torch.onnx.export(
    model, dummy, "model.onnx",
    opset_version=13,
    input_names=["input"], output_names=["output"],
    dynamic_axes={
        "input":  {0: "batch_size"},   # 第 0 维可变
        "output": {0: "batch_size"}
    }
)
```

之后推理时可输入任意 batch 大小（受限于内存）。

### 3.3 控制流模型处理
若 `forward` 有 if/while，直接导出可能失败。先转为 TorchScript：
```python
script = torch.jit.script(model)
torch.onnx.export(script, dummy, "model.onnx", ...)
```

### 3.4 导出注意事项
- 必须 `model.eval()`，否则 BN / Dropout 行为错误
- `dummy` 的 dtype 决定导出精度（一般 float32）
- 多输出时，`output_names` 写够名字即可，如 `["feat1","feat2"]`

---

## 四、验证 ONNX 模型

### 4.1 结构检查
```python
import onnx
model_onnx = onnx.load("model.onnx")
onnx.checker.check_model(model_onnx)   # 无异常则通过
```

### 4.2 数值精度对比（关键步骤）
```python
import onnxruntime as ort
import numpy as np

# PyTorch 输出
with torch.no_grad():
    ref = model(dummy).numpy()

# ONNX Runtime 输出
sess = ort.InferenceSession("model.onnx")
onnx_out = sess.run(None, {"input": dummy.numpy()})[0]

# 比较（相对误差 1e-3，绝对误差 1e-5）
np.testing.assert_allclose(ref, onnx_out, rtol=1e-3, atol=1e-5)
```

---

## 五、ONNX Runtime Python 推理（核心 API）

### 5.1 安装
```bash
pip install onnxruntime        # CPU 版本
pip install onnxruntime-gpu    # GPU 版本（带 CUDA 支持）
```

### 5.2 基本推理流程
```python
import onnxruntime as ort
import numpy as np

# 1. 创建会话
session = ort.InferenceSession("model.onnx")

# 2. 查看模型输入输出信息（可选）
print(session.get_inputs()[0].name)    # "input"
print(session.get_outputs()[0].name)   # "output"

# 3. 准备数据（numpy，float32）
data = np.random.randn(1, 3, 224, 224).astype(np.float32)

# 4. 推理
outputs = session.run(
    None,                     # 设为 None 返回所有输出
    {"input": data}           # 字典，key 与 input_names 对应
)

# 5. 取结果
result = outputs[0]           # 输出是 list，取第一个
```

### 5.3 获取指定输出
```python
outputs = session.run(
    ["output", "aux"],        # 指定需要的输出名
    {"input": data}
)
```

### 5.4 选择运行设备
```python
# CPU 推理
session = ort.InferenceSession("model.onnx", 
    providers=["CPUExecutionProvider"])

# GPU 推理
session = ort.InferenceSession("model.onnx",
    providers=["CUDAExecutionProvider"])

# 指定 GPU 编号，并支持 fallback
session = ort.InferenceSession("model.onnx",
    providers=[("CUDAExecutionProvider", {"device_id": 1}), 
               "CPUExecutionProvider"])
```

### 5.5 性能优化参数
```python
opts = ort.SessionOptions()
opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL   # 最高图优化
opts.intra_op_num_threads = 4       # 算子内并行线程数（CPU）
opts.inter_op_num_threads = 2       # 算子间并行线程数（CPU）

session = ort.InferenceSession("model.onnx", sess_options=opts,
    providers=["CPUExecutionProvider"])
```

| 图优化等级 | 说明 |
|-----------|------|
| ORT_DISABLE_ALL | 不做任何优化 |
| ORT_ENABLE_BASIC | 基础优化（常量折叠、冗余节点消除） |
| ORT_ENABLE_EXTENDED | 扩展优化（算子融合、内存规划） |
| ORT_ENABLE_ALL | 最高级别（包含扩展优化） |

一般直接使用 `ORT_ENABLE_ALL`。

---

## 六、ONNX Runtime C++ 推理环境配置

### 6.1 下载 ONNX Runtime 预编译包
从 [GitHub Releases](https://github.com/microsoft/onnxruntime/releases) 下载对应系统的包（如 `onnxruntime-linux-x64-1.18.0.tgz`），解压后目录结构：

```text
onnxruntime/
├── include/          # 头文件
│   └── onnxruntime_cxx_api.h  (及子目录)
├── lib/              # 动态库 .so / .dylib / .lib
└── LICENSE
```

需要 GPU 推理则下载带 `gpu` 字样的包（如 `onnxruntime-linux-x64-gpu-1.18.0.tgz`）。

### 6.2 CMakeLists.txt 配置
```cmake
cmake_minimum_required(VERSION 3.18)
project(ONNXDemo)

set(CMAKE_CXX_STANDARD 17)
set(CMAKE_CXX_STANDARD_REQUIRED ON)

# 指定 ONNX Runtime 路径（按实际解压位置修改）
set(ORT_ROOT "/path/to/onnxruntime")

# 头文件路径
include_directories(${ORT_ROOT}/include)

# 查找动态库
find_library(ORT_LIB onnxruntime PATHS ${ORT_ROOT}/lib REQUIRED)

add_executable(main main.cpp)
target_link_libraries(main ${ORT_LIB})

# 让可执行文件运行时能找到动态库（Linux/macOS）
set_target_properties(main PROPERTIES
    BUILD_RPATH "${ORT_ROOT}/lib"
    INSTALL_RPATH "${ORT_ROOT}/lib"
)
```

编译：
```bash
mkdir build && cd build
cmake .. && make
```

---

## 七、ONNX Runtime C++ 常用 API 详解

所有 C++ API 都在头文件 `onnxruntime_cxx_api.h` 中，命名空间 `Ort`。

### 7.1 初始化环境与会话
```cpp
#include <onnxruntime_cxx_api.h>
#include <iostream>
#include <vector>

// 创建环境（整个程序只需一个）
Ort::Env env(ORT_LOGGING_LEVEL_WARNING, "my_ort_env");

// 会话选项
Ort::SessionOptions session_options;
session_options.SetIntraOpNumThreads(4);                    // 设置线程数
session_options.SetGraphOptimizationLevel(                   // 图优化等级
    GraphOptimizationLevel::ORT_ENABLE_ALL);

// 加载模型
Ort::Session session(env, "model.onnx", session_options);
```

### 7.2 获取输入输出名称和形状
```cpp
// 获取输入/输出名称
Ort::AllocatorWithDefaultOptions allocator;
size_t num_input = session.GetInputCount();
size_t num_output = session.GetOutputCount();

std::vector<const char*> input_names(num_input);
std::vector<const char*> output_names(num_output);

for (size_t i = 0; i < num_input; i++) {
    auto name = session.GetInputNameAllocated(i, allocator);
    input_names[i] = name.get();   // 注意：name 是 unique_ptr，可直接用，但需确保生命周期
}

for (size_t i = 0; i < num_output; i++) {
    auto name = session.GetOutputNameAllocated(i, allocator);
    output_names[i] = name.get();
}

// 获取输入形状（动态维度可能为 -1）
auto type_info = session.GetInputTypeInfo(0);
auto tensor_info = type_info.GetTensorTypeAndShapeInfo();
auto input_shape = tensor_info.GetShape();
```

**注意**：`GetInputNameAllocated` 返回的是 `std::unique_ptr<char[], OrtAllocator>`，获取的指针 `name.get()` 可直接使用，但必须在 allocator 有效的情况下使用（上述写法安全，因为指针存入 vector 时 allocator 仍在作用域内）。

### 7.3 创建输入张量（关键）

使用 `Ort::Value::CreateTensor` 从已有内存创建张量。需要提供：
- `Ort::MemoryInfo`：描述内存位置（CPU 或 CUDA）和设备 ID
- `T* data`：数据指针
- `size_t size`：元素总数
- `const int64_t* shape`：形状数组
- `size_t shape_len`：形状维度

**CPU 示例**：
```cpp
std::vector<float> input_data(1 * 3 * 224 * 224, 1.0f);  // 实际数据
std::vector<int64_t> input_shape = {1, 3, 224, 224};
size_t data_size = 1 * 3 * 224 * 224;

// 创建 MemoryInfo：描述一块 CPU 内存
Ort::MemoryInfo memory_info = Ort::MemoryInfo::CreateCpu(
    OrtArenaAllocator, OrtMemTypeDefault);

// 创建输入 Value（张量）
Ort::Value input_tensor = Ort::Value::CreateTensor<float>(
    memory_info,
    input_data.data(),      // 数据指针
    data_size,              // 元素总数
    input_shape.data(),     // 形状数组
    input_shape.size()      // 形状维度
);
```

**GPU 推理时**：需分配 CUDA 内存，并使用 `Ort::MemoryInfo::Create` 传入 GPU 设备信息，一般生产环境会封装。

### 7.4 执行推理
```cpp
// 组合输入
std::vector<Ort::Value> input_values;
input_values.push_back(std::move(input_tensor));

// 运行
auto output_values = session.Run(
    Ort::RunOptions{nullptr},   // 运行选项，默认即可
    input_names.data(),         // 输入名称数组
    input_values.data(),        // 输入 Value 数组
    input_values.size(),        // 输入数量
    output_names.data(),        // 输出名称数组
    output_names.size()         // 输出数量
);
```

### 7.5 提取输出数据
```cpp
// 获取第一个输出
auto& output_tensor = output_values.front();

// 得到输出形状和元素数量
auto type_info = output_tensor.GetTensorTypeAndShapeInfo();
auto output_shape = type_info.GetShape();
size_t output_elements = type_info.GetElementCount();

// 获取数据指针（float）
float* output_data = output_tensor.GetTensorMutableData<float>();

// 使用数据
std::cout << "First element: " << output_data[0] << std::endl;
```

### 7.6 完整最小示例（C++）
```cpp
#include <onnxruntime_cxx_api.h>
#include <iostream>
#include <vector>

int main() {
    // 初始化
    Ort::Env env(ORT_LOGGING_LEVEL_WARNING, "test");
    Ort::SessionOptions opts;
    opts.SetIntraOpNumThreads(1);
    Ort::Session session(env, "model.onnx", opts);

    // 获取输入输出名
    Ort::AllocatorWithDefaultOptions allocator;
    auto input_name = session.GetInputNameAllocated(0, allocator);
    auto output_name = session.GetOutputNameAllocated(0, allocator);
    std::vector<const char*> input_names = {input_name.get()};
    std::vector<const char*> output_names = {output_name.get()};

    // 准备输入数据
    std::vector<float> data(1 * 3 * 224 * 224, 1.0f);
    std::vector<int64_t> shape = {1, 3, 224, 224};
    auto memory_info = Ort::MemoryInfo::CreateCpu(OrtArenaAllocator, OrtMemTypeDefault);
    Ort::Value input_tensor = Ort::Value::CreateTensor<float>(
        memory_info, data.data(), data.size(), shape.data(), shape.size());

    // 推理
    std::vector<Ort::Value> inputs;
    inputs.push_back(std::move(input_tensor));
    auto outputs = session.Run(Ort::RunOptions{nullptr},
                               input_names.data(), inputs.data(), 1,
                               output_names.data(), 1);

    // 取输出
    if (!outputs.empty()) {
        float* out = outputs[0].GetTensorMutableData<float>();
        auto shape_info = outputs[0].GetTensorTypeAndShapeInfo();
        std::cout << "Output size: " << shape_info.GetElementCount() << std::endl;
        std::cout << "First value: " << out[0] << std::endl;
    }
    return 0;
}
```

---

## 八、性能优化常用方法

### 8.1 图优化等级（Python / C++）
- Python：`opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL`
- C++：`opts.SetGraphOptimizationLevel(GraphOptimizationLevel::ORT_ENABLE_ALL);`

建议始终开启最高优化。

### 8.2 多线程控制（仅 CPU）
- Python：`opts.intra_op_num_threads = N`
- C++：`opts.SetIntraOpNumThreads(N);`

根据 CPU 核心数合理设置，通常 4~8 即可。

### 8.3 动态 shape
使用 `dynamic_axes` 导出模型后，ONNX Runtime 自动处理任意合法尺寸，无需额外代码。

---

## 九、常见问题速查

| 问题 | 原因 | 解决 |
|------|------|------|
| 导出时报 `Unsupported operator` | 算子不支持或 opset 太低 | 提高 `opset_version`，或先用 Script |
| 动态 batch 推理报 shape 错误 | 未设置 `dynamic_axes` | 导出时加上 `dynamic_axes` |
| 输出数值差异大 | 精度不一致或预处理差异 | 确认都为 FP32，对齐预处理 |
| C++ 加载模型失败 | 路径错误或缺少动态库 | 检查文件路径，设置 `LD_LIBRARY_PATH` |
| C++ 推理报错 "got shape ..." | 输入形状与模型不匹配 | 确保输入 shape 正确，注意 NCHW 格式 |

---

## 十、总结（最常用 API 速记）

**Python 端**：
```python
# 导出
torch.onnx.export(model, dummy, "model.onnx", 
    opset_version=17, input_names=["input"], output_names=["output"],
    dynamic_axes={"input":{0:"batch"}, "output":{0:"batch"}})

# 推理
sess = ort.InferenceSession("model.onnx")
out = sess.run(None, {"input": data})[0]
```

**C++ 端**：
```cpp
Ort::Session session(env, "model.onnx", opts);
auto input_t = Ort::Value::CreateTensor<float>(memory_info, data.data(), size, shape.data(), shape.size());
auto outputs = session.Run(RunOptions{nullptr}, input_names, &input_t, 1, output_names, 1);
float* result = outputs[0].GetTensorMutableData<float>();
```

掌握以上内容，即可完成 90% 的 ONNX 部署工作。本教程未涉及 TensorRT，如需进一步挖掘 GPU 推理极限，可单独学习 TensorRT 部署。